# IELTS — GGUF export from a trained adapter (Kaggle, disk-safe)

Use this when a training run **finished** — the LoRA adapter saved — but
the GGUF step died with `OSError: Not enough free space`. Unsloth's merge
+ F16 intermediates need ~30 GB for a 7B, over the ~20 GB
`/kaggle/working` quota. This notebook **does not retrain**: it loads your
saved adapter, then runs the export with every intermediate redirected to
`/tmp`, copying only the final ~4.7 GB quantised GGUF back into
`/kaggle/working`.

### Before you run
1. **Attach the trained adapter:** *Add Input → Notebook Output →* the run
   that saved `qwen2.5-7b-ielts-evaluator-lora` (your evaluator training
   run). It mounts under `/kaggle/input/…`; cell 2 finds it by searching
   for `adapter_config.json`, so any adapter works — not just the
   evaluator's.
2. **Accelerator = a single GPU T4**, **Internet = On** (the base model is
   downloaded to merge against). *P100 won't run Unsloth* — see the
   training notebooks for why.
3. Run All. The output folder `qwen2.5-7b-ielts-evaluator-gguf_gguf/` will
   hold `Qwen2.5-7B-Instruct.Q4_K_M.gguf` + a ready `Modelfile`.

> If the adapter isn't in the previous run's output (e.g. the disk filled
> before it saved), fall back to re-running the evaluator notebook end to
> end — its save cell is now disk-safe too.

## 0. Install Unsloth

In [ ]:
%%capture
# Unsloth gives ~2x faster QLoRA. Two Kaggle-specific gotchas: (1) the free
# Unsloth build trains on ONE GPU only, and (2) it needs a Turing-or-newer
# card — use a **T4** (compute capability 7.5). The P100 is Pascal (6.0) and
# has NO compiled Unsloth/Triton kernels -> 'no kernel image is available'.
# Fit depends on the model AND the corpus length: the generator's records
# are long (~5-7k tokens each), so it uses a 3B base; a 7B/14B OOMs a T4 at
# train time on that corpus (see the generator's section 2 for the math).
# If this install ever breaks, follow the current Kaggle snippet at
# https://github.com/unslothai/unsloth (the API used below is stable).
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 1. Load the base + your trained adapter

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob
from unsloth import FastLanguageModel

# Find the trained LoRA adapter you attached as an input (Add Input ->
# Notebook Output -> the training run). We locate the folder that holds
# adapter_config.json anywhere under /kaggle/input.
_cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if not _cfgs:
    raise FileNotFoundError(
        "No adapter found under /kaggle/input. Add the training run's "
        "OUTPUT as an input: Add Input -> Notebook Output -> pick the run "
        "that saved the *-lora folder, then re-run.")
ADAPTER_DIR = os.path.dirname(_cfgs[0])
print("Using adapter:", ADAPTER_DIR)

MAX_SEQ_LEN = 1024
# Point from_pretrained at the ADAPTER dir; Unsloth reads its base model
# from adapter_config.json, downloads it (needs Internet=On), and applies
# the trained LoRA. No get_peft_model here — the adapter is already trained.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},
)

## 2. Export GGUF (intermediates → /tmp, final → working)

In [ ]:
# The GGUF path writes a merged fp16 model, then an F16 GGUF, then the
# q4_k_m — transiently ~2x the model size (a 7B needs ~30 GB). Writing
# those into /kaggle/working (~20 GB quota) overflows it and dies with
# 'OSError: Not enough free space'. So send every intermediate to /tmp
# (the container's larger scratch) and copy back ONLY the final ~4.7 GB
# quantised GGUF + Modelfile.
import os, glob, shutil

_TMP = "/tmp/_gguf_export"
if os.path.isdir(_TMP):
    shutil.rmtree(_TMP)
model.save_pretrained_gguf(_TMP, tokenizer, quantization_method="q4_k_m")

# Unsloth writes the GGUF into a sibling '<dir>_gguf/' folder. Copy only
# the quantised file — never the ~15 GB F16 intermediate, if it lingers.
_SRC = _TMP + "_gguf"
_DST = "/kaggle/working/qwen2.5-7b-ielts-evaluator-gguf_gguf"
os.makedirs(_DST, exist_ok=True)
_all = glob.glob(f"{_SRC}/*.gguf")
_keep = [p for p in _all if "q4_k_m" in os.path.basename(p).lower()] or _all
_keep += glob.glob(f"{_SRC}/Modelfile")
for _f in _keep:
    shutil.copy(_f, _DST)
    print("kept", os.path.basename(_f), f"({os.path.getsize(_f)/1024**3:.2f} GiB)")
_final = glob.glob(f"{_DST}/*.gguf")
assert _final, "no GGUF produced — check the conversion log above"
print("Final GGUF folder:", _DST, "->", [os.path.basename(p) for p in _final])

## 3. Serve it

Download `qwen2.5-7b-ielts-evaluator-gguf_gguf/` from the output, then:
```
ollama create ielts-evaluator -f Modelfile
```
This is the **evaluator** (a separate model from the generator). Wiring it
into per-answer marking still needs the small backend addition described
in the evaluator notebook's section 6 (`EVALUATOR_SYSTEM` in
`app/llm/prompts.py`).